# 05 · Writer and Reviewer

Pattern 3: one agent **writes** a reply, a second agent **reviews** it before it goes out. If the reviewer finds a problem, the writer tries again.

```
Writer drafts ──> Reviewer checks ──approved──> send to customer
      ▲                  │
      └──── problems ────┘   (at most 3 rounds, then a human takes over)
```

Why? A model can write a friendly reply that **promises something the shop does not allow**, such as a refund outside the refund window. A second pair of eyes, comparing the draft with **what the tools actually returned**, catches it.

In [1]:
import json
from shop import client, MODEL     # the OpenAI client and the model name (see shop.py)

print("Ready")

Ready


In [2]:
from shop import run_agent, TOOL_RESULTS
from shop import LOOKUP_ORDER_TOOL, SEARCH_POLICIES_TOOL, lookup_order, search_policies

## The reviewer
It reads the draft and the **evidence** (what the tools returned), and answers with structured output: approved or not, and what is wrong.

In [3]:
from pydantic import BaseModel

class Review(BaseModel):
    reasoning: str     # first, one short sentence comparing the draft with the evidence
    approved: bool
    problems: str      # what is wrong. Empty if the draft is approved.

# The reviewer has the most important job, so it gets a stronger model.
# In a multi-agent system each agent can use a different model.
REVIEWER_MODEL = "gpt-4o"

def review_reply(draft, evidence):
    response = client.chat.completions.parse(
        model=REVIEWER_MODEL,
        messages=[
            {"role": "system", "content": (
                "You check a draft reply that a support agent wants to send to a customer. "
                "Compare it with the EVIDENCE (what the shop's tools returned). "
                "Reject the draft ONLY if (1) it claims or promises something that is not in the evidence or that "
                "contradicts it, for example promising a refund that the policy does not allow, "
                "or (2) it is rude, or (3) it is longer than 3 sentences. "
                "Do not nitpick wording. "
                "A reply that correctly reports the evidence must be approved, even if the news is bad for the customer."
            )},
            {"role": "user", "content": "EVIDENCE:\n" + evidence + "\n\nDRAFT REPLY:\n" + draft},
        ],
        response_format=Review,
    )
    return response.choices[0].message.parsed

## Does the reviewer catch a bad draft?
First a test with a draft **we wrote with a mistake**. Order A1003 was delivered 80 days ago, and refunds only work within 30 days.

In [4]:
evidence = lookup_order("A1003") + "\n" + search_policies("refund window")

bad_draft = "Of course, I am so sorry! I have refunded your coffee grinder in full. The money will arrive in 5 days."

review = review_reply(bad_draft, evidence)
print("Approved:", review.approved)
print("Problems:", review.problems)

Approved: False
Problems: The draft claims a refund has been issued which contradicts the returns policy stating refunds are only possible within 30 days of delivery.


The reviewer rejects it, and says why.

## The writer
The writer is an agent with the Orders and Policy tools. Its instructions make it eager to please, which is exactly when mistakes happen.

In [5]:
writer_agent = {
    "name": "Writer",
    "instructions": (
        "You are a friendly support agent who wants every customer to leave happy. "
        "Use lookup_order for order details and search_policies for shop rules. "
        "Write the reply to the customer in at most 3 sentences."
    ),
    "tools": [LOOKUP_ORDER_TOOL, SEARCH_POLICIES_TOOL],
    "functions": {"lookup_order": lookup_order, "search_policies": search_policies},
}

## The write, review, revise loop
`MAX_ROUNDS` is a safety limit. After 3 failed rounds a **human** takes over instead of looping forever.

In [6]:
MAX_ROUNDS = 3

customer_message = "I'm really upset. My coffee grinder (order A1003) broke after 3 months. Please just refund it, I'm sure you can make an exception!"

TOOL_RESULTS.clear()                 # start with an empty evidence log
message_for_writer = customer_message
approved = False
final_reply = ""

for round_number in range(1, MAX_ROUNDS + 1):
    print("--- Round", round_number, "---")

    # 1. The writer drafts a reply
    draft = run_agent(writer_agent, message_for_writer)
    print("  Draft   :", draft)

    # 2. The reviewer checks it against what the tools returned
    evidence = "\n".join(TOOL_RESULTS)
    review = review_reply(draft, evidence)

    if review.approved:
        print("  Reviewer: approved")
        approved = True
        final_reply = draft
        break

    # 3. Not approved: tell the writer what to fix, then go round again
    print("  Reviewer: rejected -", review.problems)
    message_for_writer = (
        customer_message
        + "\n\nYour previous draft was rejected. Problems: " + review.problems
        + "\nWrite a corrected reply."
    )

print()
if approved:
    print("Sent to the customer:", final_reply)
else:
    print("Not approved after", MAX_ROUNDS, "rounds: passed to a human agent.")

--- Round 1 ---


  [Writer] lookup_order({'order_id': 'A1003'})


  [Writer] search_policies({'question': 'refund policy'})


  Draft   : I understand how frustrating this situation is. Unfortunately, our refund policy allows returns only within 30 days after delivery, but since your coffee grinder is faulty, you can still return it under warranty for a replacement. Please let me know if you'd like to initiate that process!


  Reviewer: approved

Sent to the customer: I understand how frustrating this situation is. Unfortunately, our refund policy allows returns only within 30 days after delivery, but since your coffee grinder is faulty, you can still return it under warranty for a replacement. Please let me know if you'd like to initiate that process!


Depending on the run, the writer may get it right straight away or need a second round. Either way, **nothing reaches the customer until the reviewer approves it**.

## Recap
* **Writer and reviewer**: one agent drafts, another checks, and the draft is revised until it passes.
* The reviewer checks the draft against **evidence** (what the tools returned), not against its own opinion.
* A round limit and a **human fallback** stop the loop from running forever.
* The reviewer is an extra model call, so use it where a mistake is costly, such as refunds, legal wording or medical advice.
* In this notebook the writer's tools only **read** information, so running them again each round is safe. In the real-life script the Refund agent can **record a refund**, so there only the **wording** is rewritten and the tools are never run twice.

In the real-life script, `support_desk.py`, all three patterns work together: a **router** hands the email to a specialist, the specialist writes the reply, and a **reviewer** checks it before it is sent.